# 15 · The breasts refined from close-ups

Notebook 12 built an avatar from one video; its breasts are what the body fit could tell from
full-body frames, a few dozen texels across. Close-ups of them are far more common: notebook 14
harvests them from many videos of the same actor. Here they refine the avatar's breasts:

| Step | What | Module |
| --- | --- | --- |
| Close-ups | each harvested frame cropped around the breasts, with NSFW-API's breast, areola and nipple masks, Sapiens2's body parts and normals, ViTPose's keypoints | `regions.closeups` |
| Shape | Anny's breast controls and smooth offsets in the breast region, fitted to the upright close-ups; each close-up placed in its camera | `regions.breasts` |

Each step reads the `.avatar` file notebook 12 wrote and writes a refined one, so the eyes,
face hair and light baked into it carry over.

In [ ]:
import csv
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from avatar_splats.body_models.anny import AnnyBody
from avatar_splats.body_models.hybrid import Avatar
from avatar_splats.config import load_config
from avatar_splats.export.avatar_file import read_avatar, write_avatar
from avatar_splats.preprocess.prepare import NOT_THEM
from avatar_splats.regions.breasts import (
    BREAST_CHANGES,
    BREAST_PHENOTYPES,
    BreastFit,
    FitSettings,
    fit_breasts,
    render_closeups,
    with_breasts,
)
from avatar_splats.regions.closeups import Closeups, gather_closeups

CONFIG = "../configs/default.yaml"
config = load_config(CONFIG)
device = "cuda"

## Settings

`NAME` is notebook 12's avatar (`outputs/video_avatar/<NAME>/`, `outputs/export/<NAME>.avatar`),
`HARVEST` notebook 14's run whose close-ups are of the same actor. A close-up shapes the breasts
only if her chest leans at most `max_tilt` from upright (they sag otherwise), her likeness to the
actor is known and at least `min_identity`, and it was placed well (`min_iou`).

In [ ]:
NAME = "water"
HARVEST = "live-performance"
VIDEO_ROOT = Path("../examples")  # harvest video "folder/name" is VIDEO_ROOT / "folder/name.mp4"
REUSE = {"closeups", "shape"}  # steps that start from an earlier run's results, where stored
KINDS = ("kept", "passed")  # the harvest's verdicts of breast close-ups taken

FIT = FitSettings(
    render_size=256,
    pose_steps=300,
    shape_steps=600,
    learning_rate=0.01,
    smoothing_steps=8,  # rounds of averaging the offsets: their smoothness
    max_tilt=30.0,
    min_identity=0.35,
    min_iou=0.6,
    weights={
        "outline": 4.0,
        "normals": 1.0,
        "nipples": 20.0,
        "joints": 2.0,
        "pose": 0.1,
        "offsets": 1e-5,  # per square millimetre
        "controls": 0.01,
    },  # fmt: skip
)

avatar_dir = config.outputs / "video_avatar" / NAME
out_dir = config.outputs / "regions" / NAME
out_dir.mkdir(parents=True, exist_ok=True)
avatar_file = read_avatar(config.outputs / "export" / f"{NAME}.avatar")
arrays = dict(avatar_file.arrays, head_start=avatar_file.metadata["head_start"])
bone_labels = list(avatar_file.metadata["bones"])
body_fit = np.load(avatar_dir / "body.npz")
fitted = Avatar.load(avatar_dir / "avatar.npz")
body = AnnyBody().to(device)

## Close-ups

The harvest's prepared breast close-ups (`KINDS`), from every video of the run. A video's field
of view is the body fit's for notebook 12's own video, else 35°: a close-up's distance is free
in the fit, so a wrong lens bends perspective a little but scales nothing.

In [ ]:
path = out_dir / "closeups.npz"
if "closeups" in REUSE and path.exists():
    closeups = Closeups.load(path)
else:
    with open(config.outputs / "harvest" / HARVEST / "harvest.csv") as file:
        records = [
            r for r in csv.DictReader(file)
            if r["stage"] == "prepared" and r.get("breasts_verdict") in KINDS
        ]  # fmt: skip
    videos = sorted({r["video"] for r in records})
    region_names = {
        int(k): v
        for k, v in json.loads(Path("../assets/nsfw/pakaho-nsfw-v3.json").read_text()).items()
    }
    own = next((v for v in videos if v.split("/")[-1] == NAME), None)
    closeups = gather_closeups(
        records, {v: VIDEO_ROOT / f"{v}.mp4" for v in videos},
        {own: float(body_fit["fov_y_degrees"])} if own else {}, region_names,
        Path("../assets/nsfw"), "facebook/sapiens2-seg-1b", "facebook/sapiens2-normal-1b", device,
    )  # fmt: skip
    closeups.save(path)
found = np.isfinite(closeups.nipples[:, :, 0]).sum(1)
print(
    f"{len(closeups.keys)} close-ups from {len(set(closeups.videos))} videos; "
    f"nipples found: {np.bincount(found, minlength=3).tolist()} (none, one, two)"
)

In [ ]:
def masks_view(i):
    image = closeups.images[i].copy()
    image[closeups.breasts[i]] = (0.5 * image[closeups.breasts[i]] + (0, 90, 0)).astype(np.uint8)
    image[closeups.areolae[i]] = (255, 0, 0)
    for x, y in closeups.nipples[i]:
        if np.isfinite(x):
            cv2.circle(image, (int(x), int(y)), 6, (255, 255, 0), 2)
    return image


def normals_view(normals):
    return ((normals.astype(np.float32) * [1, -1, -1] + 1) * 127.5).astype(np.uint8)


colours = np.random.default_rng(2).integers(0, 255, (256, 3)).astype(np.uint8)
colours[NOT_THEM] = 0
shown = np.linspace(0, len(closeups.keys) - 1, 4).astype(int)
fig, axes = plt.subplots(len(shown), 4, figsize=(14, 3.6 * len(shown)))
for row, i in zip(axes, shown, strict=True):
    for ax, image, title in zip(
        row,
        (closeups.images[i], masks_view(i), normals_view(closeups.normals[i]),
         colours[closeups.labels[i]]),
        (closeups.keys[i], "NSFW-API: breasts, areolae, nipples", "Sapiens2 normals",
         "Sapiens2 body parts"),
        strict=True,
    ):  # fmt: skip
        ax.imshow(image)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## The breasts' shape

First every close-up is placed with the avatar's breasts (its turn and shift in the camera, the
spine, clavicles and upper arms turned), then shape and places are fitted together, the shape
by the upright close-ups only. Size stays the avatar's: a close-up's distance is free, and the
torso's outline around the breasts sets the scale.

In [ ]:
path = out_dir / "breasts.npz"
if "shape" in REUSE and path.exists() and np.array_equal(BreastFit.load(path).keys, closeups.keys):
    fit = BreastFit.load(path)
else:
    fit = fit_breasts(
        body, arrays, fitted.rest_vertices[: fitted.head_start], body_fit["phenotype"],
        body_fit["local_changes"], bone_labels, closeups, FIT, device,
    )  # fmt: skip
    fit.save(path)

print(
    f"{fit.shaping.sum()} of {len(fit.keys)} close-ups shaped the breasts; "
    f"her chest's lean: median {np.median(fit.tilt):.0f}°, "
    f"90 % under {np.percentile(fit.tilt, 90):.0f}°"
)
for name, before, after in zip(
    (*BREAST_PHENOTYPES, *BREAST_CHANGES), fit.controls_before, fit.controls, strict=True
):
    print(f"  {name:24} {before:6.3f} -> {after:6.3f}")
print(f"the surface moved up to {np.linalg.norm(fit.change, axis=1).max() * 1000:.1f} mm")
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, (name, before, after, unit) in zip(axes, (
    ("torso outline IoU", fit.iou_before, fit.iou_after, ""),
    ("normals over the breasts", fit.normal_before, fit.normal_after, "degrees"),
    ("nipples", fit.nipple_before, fit.nipple_after, f"pixels of {FIT.render_size}"),
), strict=True):  # fmt: skip
    shaping = fit.shaping
    ax.hist([before[shaping], after[shaping]], bins=20, label=["avatar's", "fitted"])
    ax.set_title(
        f"{name}: median {np.median(before[shaping]):.3g} -> {np.median(after[shaping]):.3g}"
    )
    ax.set_xlabel(unit)
    ax.legend()
plt.tight_layout()
plt.show()

The close-ups with the avatar's breasts and the fitted ones, shaded, beside Sapiens2's normals:
the most improved, a middling one and the least improved. Cyan: the fitted torso's outline;
magenta: the model's nipples; green rings: the found ones.

In [ ]:
size = closeups.images.shape[1]
before = render_closeups(body, arrays, bone_labels, fit, closeups, size, False, device)
after = render_closeups(body, arrays, bone_labels, fit, closeups, size, True, device)


def shaded(image, i):
    light = np.clip(image["normals"][i] @ np.array([0.3, -0.4, -0.85]), 0, 1)
    return (np.stack([light] * 3, -1) * 230 * image["coverage"][i][..., None]).astype(np.uint8)


def overlay(i):
    image = closeups.images[i].copy()
    edge = cv2.Canny((after["coverage"][i] > 0.5).astype(np.uint8) * 255, 50, 150) > 0
    image[edge] = (0, 255, 255)
    for x, y in after["nipples"][i]:
        cv2.circle(image, (int(x), int(y)), 4, (255, 0, 255), 2)
    for x, y in closeups.nipples[i]:
        if np.isfinite(x):
            cv2.circle(image, (int(x), int(y)), 8, (0, 255, 0), 1)
    return image


gain = np.where(fit.shaping, fit.normal_before - fit.normal_after, np.nan)
order = np.argsort(-np.nan_to_num(gain, nan=-np.inf))[: fit.shaping.sum()]
shown = [order[0], order[len(order) // 2], order[-1]]
fig, axes = plt.subplots(len(shown), 4, figsize=(14, 3.6 * len(shown)))
for row, i in zip(axes, shown, strict=True):
    for ax, image, title in zip(
        row, (overlay(i), shaded(before, i), shaded(after, i), normals_view(closeups.normals[i])),
        (closeups.keys[i], f"avatar's: {fit.normal_before[i]:.1f}°",
         f"fitted: {fit.normal_after[i]:.1f}°", "Sapiens2 normals"),
        strict=True,
    ):  # fmt: skip
        ax.imshow(image)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

How far the surface moved, on her rest pose from the front and the side, in millimetres.

In [ ]:
rest = arrays["rest_vertices"][: arrays["head_start"]]
moved = np.linalg.norm(fit.change, axis=1) * 1000
near = fit.weights > 0.01
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
views = ((0, 2, 1, "front (x, z)"), (1, 2, 0, "side (y, z)"))  # axes drawn, depth axis
for ax, (a, b, depth, label) in zip(axes, views, strict=True):
    shown = np.argsort(-rest[near, depth])  # the far ones first, the near ones over them
    points = ax.scatter(
        rest[near, a][shown], rest[near, b][shown], c=moved[near][shown], s=6, cmap="magma"
    )
    ax.set_aspect("equal")
    ax.set_title(label)
fig.colorbar(points, ax=axes, label="mm")
plt.show()

## The refined avatar

`outputs/export/<NAME>-regions.avatar`: notebook 12's avatar with the fitted breasts; the
viewer opens it as it does the others.

In [ ]:
refined = with_breasts(avatar_file, fit)
path = config.outputs / "export" / f"{NAME}-regions.avatar"
write_avatar(path, refined)
print(f"wrote {path} ({path.stat().st_size / 1e6:.0f} MB)")